# 🖐 Control de Volumen con Landmarks de Mano
### Laboratorio de Tecnologías de la Imagen Digital — IFTS N° 24

Este notebook demuestra el uso de **MediaPipe Hand Landmarker** para detectar puntos clave de la mano en tiempo real y traducir esa información gestual en una acción concreta: **subir y bajar el volumen del sistema**.

---

## Concepto

La mano tiene **21 landmarks** numerados. Vamos a usar la **posición vertical (eje Y) de la muñeca** como control:

- Mano **arriba** → volumen **alto**
- Mano **abajo** → volumen **bajo**

```
  4
  |   8  12  16  20
  3   |   |   |   |
  |   7  11  15  19
  2   |   |   |   |
  |   6  10  14  18
  1   |   |   |   |
   \  5   9  13  17
    \ |   |   |   |
     [0: MUÑECA]      ← usamos este punto
```

El landmark `0` (muñeca) tiene coordenada `y` normalizada entre `0.0` (arriba de la imagen) y `1.0` (abajo). Invertimos esa escala para mapearla a volumen (0%–100%).

## Paso 1 — Preparar el entorno y el modelo

Instalá las dependencias desde la terminal con `uv sync` (o `uv add ...`) y luego ejecutá esta celda **una sola vez** para descargar el modelo pre-entrenado.

In [1]:
# Descarga del modelo Hand Landmarker
import urllib.request, os

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
MODEL_PATH = "hand_landmarker.task"

if not os.path.exists(MODEL_PATH):
    print("Descargando modelo...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("✓ Modelo descargado:", MODEL_PATH)
else:
    print("✓ Modelo ya disponible:", MODEL_PATH)

✓ Modelo ya disponible: hand_landmarker.task


## Paso 2 — Funciones auxiliares

Definimos las funciones para:
1. Detectar la mano
2. Mapear la posición Y al rango de volumen
3. Ejecutar el cambio de volumen en el sistema operativo, incluido Windows con `pycaw`

In [2]:
import mediapipe as mp
import cv2
import subprocess
import platform

# --- Configuración del Hand Landmarker ---

BaseOptions         = mp.tasks.BaseOptions
HandLandmarker      = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode   = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
)


# --- Control de volumen según el SO ---

OS = platform.system()  # 'Darwin' = macOS, 'Linux', 'Windows'
print(f"Sistema operativo detectado: {OS}")

WINDOWS_VOLUME = None
VOLUME_RUNTIME_WARNING_SHOWN = False

if OS == 'Windows':
    try:
        from comtypes import CoInitialize
        from pycaw.pycaw import AudioUtilities

        CoInitialize()
        devices = AudioUtilities.GetSpeakers()
        WINDOWS_VOLUME = devices.EndpointVolume
        print("✓ Control de volumen de Windows inicializado.")
    except Exception as e:
        WINDOWS_VOLUME = None
        print(f"Aviso: no se pudo inicializar pycaw ({e})")

def set_volume(level: int):
    """Establece el volumen del sistema. level: 0–100"""
    global VOLUME_RUNTIME_WARNING_SHOWN
    level = max(0, min(100, level))
    try:
        if OS == 'Darwin':  # macOS
            subprocess.run(
                ['osascript', '-e', f'set volume output volume {level}'],
                capture_output=True
            )
        elif OS == 'Linux':
            subprocess.run(
                ['amixer', '-q', 'sset', 'Master', f'{level}%'],
                capture_output=True
            )
        elif OS == 'Windows':
            if WINDOWS_VOLUME is None:
                return
            WINDOWS_VOLUME.SetMute(0, None)
            WINDOWS_VOLUME.SetMasterVolumeLevelScalar(level / 100.0, None)
    except Exception as e:
        if not VOLUME_RUNTIME_WARNING_SHOWN:
            print(f"Aviso: no se pudo aplicar el volumen del sistema ({e})")
            VOLUME_RUNTIME_WARNING_SHOWN = True


# --- Mapeo de coordenada Y a volumen ---
# y=0.0 es la parte superior de la imagen → volumen alto
# y=1.0 es la parte inferior               → volumen bajo

def y_to_volume(y: float) -> int:
    """Convierte coordenada y normalizada [0,1] a porcentaje de volumen [0,100]."""
    volume = int((1.0 - y) * 100)
    return max(0, min(100, volume))


# --- Visualización: dibuja landmarks sobre el frame ---

LANDMARK_COLOR   = (0, 255, 120)    # verde
CONNECTION_COLOR = (255, 255, 255)  # blanco
WRIST_COLOR      = (0, 120, 255)    # naranja para la muñeca

# Conexiones estándar de MediaPipe (pares de índices)
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17)
]

def draw_landmarks(frame, landmarks):
    h, w = frame.shape[:2]
    pts = [(int(lm.x * w), int(lm.y * h)) for lm in landmarks]

    # Conexiones
    for a, b in HAND_CONNECTIONS:
        cv2.line(frame, pts[a], pts[b], CONNECTION_COLOR, 1, cv2.LINE_AA)

    # Puntos
    for i, (px, py) in enumerate(pts):
        color = WRIST_COLOR if i == 0 else LANDMARK_COLOR
        radius = 7 if i == 0 else 4
        cv2.circle(frame, (px, py), radius, color, -1, cv2.LINE_AA)

    return pts


def draw_hud(frame, volume: int, hand_detected: bool):
    h, w = frame.shape[:2]

    # Barra de volumen lateral
    bar_x, bar_y, bar_h, bar_w = w - 50, 30, h - 60, 30
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h),
                  (60, 60, 60), -1)
    fill_h = int(bar_h * volume / 100)
    fill_color = (0, 200 + int(55 * volume/100), 100)
    cv2.rectangle(frame,
                  (bar_x, bar_y + bar_h - fill_h),
                  (bar_x + bar_w, bar_y + bar_h),
                  fill_color, -1)
    cv2.rectangle(frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h),
                  (180, 180, 180), 1)

    # Texto volumen
    cv2.putText(frame, f"{volume}%",
                (bar_x - 5, bar_y + bar_h + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(frame, "VOL",
                (bar_x + 2, bar_y - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1, cv2.LINE_AA)

    # Estado
    status = "MANO DETECTADA" if hand_detected else "Sin deteccion"
    color_status = (0, 255, 120) if hand_detected else (80, 80, 80)
    cv2.putText(frame, status, (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, color_status, 1, cv2.LINE_AA)

    # Instrucciones
    cv2.putText(frame, "Mano arriba = volumen alto", (15, h - 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1, cv2.LINE_AA)
    cv2.putText(frame, "Mano abajo  = volumen bajo", (15, h - 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1, cv2.LINE_AA)
    cv2.putText(frame, "[Q] para salir", (15, h - 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (120, 120, 120), 1, cv2.LINE_AA)


print("✓ Funciones auxiliares cargadas.")

Sistema operativo detectado: Windows
✓ Control de volumen de Windows inicializado.
✓ Funciones auxiliares cargadas.


## Paso 3 — Loop principal

Ejecutá esta celda para iniciar la webcam. Una ventana de OpenCV se abrirá con la cámara en tiempo real.

- **Mové la mano** dentro del cuadro
- La barra lateral muestra el volumen en tiempo real
- **Presioná `Q`** para cerrar

> ℹ️ Si la cámara no abre o el volumen no cambia, cerrá otras apps que estén usando la webcam o el audio. En macOS, aceptá los permisos de cámara si el sistema los solicita.

In [3]:
# ─── LOOP PRINCIPAL ────────────────────────────────────────────────────────────

cap = cv2.VideoCapture(0)  # 0 = cámara por defecto

# Suavizado exponencial del volumen para evitar saltos bruscos
SMOOTHING = 0.15  # valor entre 0 (muy lento) y 1 (sin suavizado)
smooth_volume = 50  # volumen inicial

# Control de cuántas veces por segundo se aplica el volumen al SO
# (evitar llamadas excesivas a osascript/amixer)
APPLY_EVERY_N_FRAMES = 5
frame_count = 0
last_applied_volume = None

print("Cámara activa. Presioná Q en la ventana para cerrar.")

try:
    if not cap.isOpened():
        raise RuntimeError("No se pudo acceder a la cámara. Verificá los permisos.")

    with HandLandmarker.create_from_options(options) as landmarker:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)  # espejo horizontal (más intuitivo)

            # Convertir a MediaPipe Image
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

            # Detección
            result = landmarker.detect(mp_image)

            hand_detected = len(result.hand_landmarks) > 0
            volume_target = smooth_volume  # mantener si no hay mano

            if hand_detected:
                landmarks = result.hand_landmarks[0]  # primera mano
                wrist_y   = landmarks[0].y             # landmark 0 = muñeca

                volume_target = y_to_volume(wrist_y)
                draw_landmarks(frame, landmarks)

            # Suavizado exponencial
            smooth_volume = smooth_volume + SMOOTHING * (volume_target - smooth_volume)
            displayed_volume = int(smooth_volume)

            # Aplicar volumen al SO cada N frames
            frame_count += 1
            if frame_count % APPLY_EVERY_N_FRAMES == 0 and displayed_volume != last_applied_volume:
                set_volume(displayed_volume)
                last_applied_volume = displayed_volume

            # HUD
            draw_hud(frame, displayed_volume, hand_detected)

            cv2.imshow('Control de Volumen — MediaPipe Hands', frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("Saliendo...")
                break

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("✓ Sesión cerrada.")

Cámara activa. Presioná Q en la ventana para cerrar.
Saliendo...
✓ Sesión cerrada.


---

## Para explorar en clase

Una vez que el ejemplo base funciona, podés proponer estas variaciones como ejercicios:

### Variación 1 — Usar la distancia entre dedos
En lugar de la posición de la muñeca, usar la **distancia entre el pulgar (landmark 4) y el índice (landmark 8)** como control. Es el gesto clásico de "pinch-to-zoom".

```python
import math

def distance(lm_a, lm_b):
    return math.sqrt((lm_a.x - lm_b.x)**2 + (lm_a.y - lm_b.y)**2)

d = distance(landmarks[4], landmarks[8])  # pulgar - índice
# d ≈ 0.0 (dedos juntos) a 0.4 (dedos separados)
volume_target = int((d / 0.4) * 100)
```

### Variación 2 — Usar ambas manos
Cambiar `num_hands=2` en las opciones y usar una mano para volumen, la otra para brillo.

### Variación 3 — Contar dedos levantados
Detectar si cada dedo está extendido (comparando la punta con la articulación media) y mapear el conteo (0–5) a valores discretos de volumen: 0%, 20%, 40%, 60%, 80%, 100%.

---

## Referencias

- [MediaPipe Hand Landmarker — Documentación oficial](https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker)
- [Índice de landmarks de la mano](https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker#hand_landmark_model_bundle)
- Farocki, H. (2004). *Imágenes operativas*. — Para la discusión sobre visión computacional y agencia de la imagen.